In [ ]:
import random
from datasets import load_dataset
from typing import List, Dict, Any
import time

# =================================================================
# 🤖 학습 목표: AI 대화 데이터 분석가를 위한 데이터 탐험!
# 🚀 과제: 복잡한 LLM 대화 데이터셋에서 핵심 정보를 추출하는 법을 배웁니다.
# 📝 데이터셋명: nayohan/Magpie-Air-MT-300K-v0.1-ko
# 📜 설명: 이 데이터셋은 다양한 LLM(거대언어모델)들이 생성한 대화 쌍(Instruction/Conversation)으로 이루어져 있습니다.
# 이 데이터는 LLM이 '지시를 잘 따르도록' 훈련(Alignment)하는 데 사용됩니다.
# 초보자가 복잡한 리스트 구조(conversations)에서 실제 '질문'과 '답변'을 분리하는 실전 기술을 익혀봅시다!
# =================================================================

# --- 설정 변수 ---
DATASET_NAME = "nayohan/Magpie-Air-MT-300K-v0.1-ko"
SPLIT_NAME = 'train'
SAMPLE_COUNT = 50  # 전체 데이터셋 중 테스트할 샘플 수 (너무 많으면 시간이 오래 걸려요!)

# -----------------------------------------------------------------
# 💡 1단계: 데이터셋 로드 전략 구축 (Streaming vs. Full Load)
# -----------------------------------------------------------------

print("=================================================================")
print("💡 1단계: 데이터 로딩 전략을 결정합니다.")
print(f"📚 데이터셋: {DATASET_NAME} (한국어 대화 튜닝 데이터)")
print("=================================================================")

dataset = None
try:
    # 먼저 스트리밍 모드로 로드 시도를 합니다. (가장 빠르고 메모리 효율적!)
    print(f"✅ 스트리밍(streaming=True)으로 '{SPLIT_NAME}' 스플릿 로드를 시도합니다...")
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print("✨ 스트리밍 모드 로드 성공! 메모리 걱정 없이 데이터를 빠르게 탐색할 수 있어요.")

except Exception as e:
    # 스트리밍 로드가 실패했을 경우 (예: 네트워크 문제, 특정 환경 설정 문제)
    print(f"⚠️ 스트리밍 로드 실패 감지. 에러: {e}")
    print("🔎 일반(Non-Streaming) 모드로 소량의 데이터를 로드하여 진행합니다.")
    try:
        # 스트리밍 대신 일반 모드로 로드합니다. (더 많은 메모리를 사용하지만 안정적입니다)
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME)
    except Exception as e_fallback:
        print(f"🛑 데이터셋 로드에 실패했습니다. {e_fallback}")
        exit()


# -----------------------------------------------------------------
# ⚙️ 2단계: 샘플러 정의 및 데이터 접근 방식 통일
# -----------------------------------------------------------------

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)이므로 이 패턴을 사용해야 해요!
    print("\n✨ 스트리밍 패턴 적용: .take() 메서드를 사용하여 샘플러를 만듭니다.")
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)의 경우
    print("\n✨ 일반 패턴 적용: list() 변환을 통해 샘플러를 만듭니다.")
    # 일반 데이터셋은 .take()가 있어도 list(dataset.take(N))이 가장 안전합니다.
    # 하지만 여기서는 list(dataset.select(indices)) 패턴을 사용하지 않으므로,
    # 굳이 반복문으로 처리하거나 list(dataset) 전체를 처리해야 합니다.
    # 안전을 위해, 여기서는 list() 변환을 통해 샘플을 확보합니다.
    try:
        dataset_list = list(dataset)
        sampled_dataset_iterator = dataset_list[:SAMPLE_COUNT]
    except MemoryError:
        print("🛑 메모리 오류: 샘플 크기를 더 줄여서 진행해 주세요.")
        exit()

# 샘플 데이터를 실제로 list로 가져옵니다. (가장 안전한 데이터 처리 방식)
sample_data_list: List[Dict[str, Any]] = list(sampled_dataset_iterator)
print(f"✅ 최종적으로 {min(SAMPLE_COUNT, len(sample_data_list))}개의 샘플을 확보하여 실습을 진행합니다.")

# -----------------------------------------------------------------
# 🔮 3단계: 핵심 실습 - 대화(Conversation) 구조 분석
# -----------------------------------------------------------------

def extract_conversation_pair(sample: Dict[str, Any]) -> tuple[str, str]:
    """
    복잡한 'conversations' 리스트에서 프롬프트(Input)와 응답(Output)을 추출합니다.
    이 함수가 오늘 실습의 핵심 목표예요!
    """
    conversations: List[Dict[str, str]] = sample.get('conversations', [])
    
    if not conversations:
        return "", "" # 대화가 없으면 빈 문자열 반환
    
    # 턴(turn)을 순회하며 대화의 흐름을 만듭니다.
    prompt_turns = []
    response_turns = []
    
    for i, turn in enumerate(conversations):
        from_role = turn.get('from', '').strip()
        value = turn.get('value', '').strip()
        
        if not value:
            continue

        # 대화의 주도권을 가진 역할(예: 'user', 'system')을 찾습니다.
        if from_role.lower() == 'user' or from_role.lower() == 'system':
            prompt_turns.append(value)
        
        # 응답 역할을 가진 역할(예: 'assistant')을 찾거나, 모든 내용을 응답으로 간주합니다.
        # 여기서는 가장 마지막 발화를 응답으로 가정해 보겠습니다.
        if from_role.lower() == 'assistant':
            response_turns.append(value)
            
    # 전체 프롬프트 (사용자의 모든 질문을 합침)
    full_prompt = "\n".join(prompt_turns)
    # 최종 답변 (가장 마지막 답변을 응답으로 간주)
    final_response = "\n".join(response_turns) if response_turns else ""
    
    return full_prompt, final_response

print("\n\n=================================================================")
print("✨ 🚀 3단계 실습: 대화 분석가 모드 ON! (데이터 구조 파헤치기)")
print("=================================================================")

extracted_prompts: List[str] = []
extracted_responses: List[str] = []
model_counts: Dict[str, int] = {}
turn_count_list: List[int] = []

for i, sample in enumerate(sample_data_list):
    # 1. 대화쌍 추출 (오늘의 핵심 기술!)
    prompt, response = extract_conversation_pair(sample)
    
    # 2. 분석 리스트에 저장
    extracted_prompts.append(prompt)
    extracted_responses.append(response)
    
    # 3. 통계 분석을 위한 정보 추출
    model_name = sample.get('model', 'Unknown')
    model_counts[model_name] = model_counts.get(model_name, 0) + 1
    
    # 대화의 턴(list length)을 기록합니다.
    turn_count = len(sample.get('conversations', []))
    turn_count_list.append(turn_count)


# -----------------------------------------------------------------
# 📊 4단계: 결과 분석 및 탐험 보고서 작성 (Quantitative Analysis)
# -----------------------------------------------------------------

print("\n\n=================================================================")
print("🔬 4단계: 탐험 보고서 작성 (Data Insights)")
print("=================================================================")

# 1. 모델 출현 빈도 분석
print("\n[🔎 1. 데이터 출처 모델 분석 (Source Model Distribution)]")
if model_counts:
    for model, count in model_counts.items():
        print(f"   - 모델 '{model}': 총 {count}개의 샘플에서 사용되었습니다.")
else:
    print("   - 분석할 모델 정보가 충분하지 않습니다.")

# 2. 대화 턴 수 분포 분석
print("\n[📊 2. 대화 턴 수 분포 분석 (Conversation Length)]")
if turn_count_list:
    from collections import Counter
    turn_counts = Counter(turn_count_list)
    
    # 가장 흔한 턴 수를 찾기
    most_common = turn_counts.most_common(3)
    
    print(f"   - 총 {len(turn_count_list)}개 샘플의 대화 길이를 분석했습니다.")
    print(f"   - 가장 많이 발견된 대화 턴 수:")
    for turns, count in most_common:
        print(f"     ▶️ {turns} 턴 (턴 수): {count}개 샘플")
else:
    print("   - 대화 턴 수 데이터를 얻을 수 없었습니다.")


# 3. 예시 결과 확인
print("\n[⭐ 3. 실제 추출된 결과 샘플 3가지 확인 (The Magic!)]")
for i in range(min(3, len(extracted_prompts))):
    print("-" * 40)
    print(f"   [샘플 {i+1}]")
    print(f"   👤 추출된 프롬프트 (Prompt): {extracted_prompts[i][:80]}...")
    print(f"   🤖 추출된 답변 (Response): {extracted_responses[i][:80]}...")

print("\n\n=================================================================")
print("🎉🎉🎉 실습 완료! 🎉🎉🎉")
print("축하합니다! 이제 복잡한 딥러닝 대화 데이터셋의 구조를 읽고,")
print("필요한 정보(Prompt, Response)를 깔끔하게 추출하는 능력을 갖추었습니다.")
print("이것이 바로 실전 AI 데이터 전처리 능력입니다!")
print("=================================================================")